# Saudi Real Estate Market Analysis

An analysis of recorded real estate sales in Saudi Arabia during Q1 and Q2 2024. The quarterly CSV files are checked, combined, and prepared for the Power BI market overview dashboard.

Source: [Saudi Real Estate Data — MOJ sales](https://github.com/civillizard/Saudi-Real-Estate-Data/tree/main/moj/sales).

## Source files

The two quarterly files are loaded separately to check their structure and record counts before combining them.

In [ ]:
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)

df_q1 = pd.read_csv("../data/MOJ-Sales-2024-Q1.csv")
df_q2 = pd.read_csv("../data/MOJ-Sales-2024-Q2.csv")

print("Q1:", df_q1.shape)
print("Q2:", df_q2.shape)

## Initial data checks

The first checks cover column names, data types, and missing values in the Q1 and Q2 files.

In [ ]:
df_q1.head()

In [ ]:
df_q1.info()

In [ ]:
df_q1.isna().sum()

In [ ]:
df_q2.isna().sum()

## Empty records

One completely empty row is removed from Q2. Other records are retained at this stage.

In [ ]:
df_q1 = df_q1.dropna(how="all").copy()
df_q2 = df_q2.dropna(how="all").copy()

print("Q1 after cleanup:", len(df_q1))
print("Q2 after cleanup:", len(df_q2))

## Combining Q1 and Q2

A `quarter` column identifies the source period of each record in the combined dataset.

In [ ]:
df_q1["quarter"] = "Q1"
df_q2["quarter"] = "Q2"

df = pd.concat([df_q1, df_q2], ignore_index=True)

df.head()

## Price and area

Price and area are converted to numeric columns so totals, averages, and comparisons can be calculated.

In [ ]:
df["السعر"] = pd.to_numeric(df["السعر"].astype(str).str.replace(",", "", regex=False),errors="coerce")

df["المساحة"] = pd.to_numeric(df["المساحة"].astype(str).str.replace(",", "", regex=False),errors="coerce")

df[["السعر", "المساحة"]].describe()

## Invalid values

Missing, zero, and negative values are checked before using price and area in the analysis.

In [ ]:
print("Missing values:\n")
print(df[["السعر", "المساحة"]].isna().sum())

print("Prices at or below zero:", (df["السعر"] <= 0).sum())
print("Areas at or below zero:", (df["المساحة"] <= 0).sum())

## Records used in the analysis

Records with missing or non-positive price or area are excluded from the cleaned table.

In [ ]:
df_clean = df.dropna(subset=["السعر", "المساحة"]).copy()

df_clean = df_clean[(df_clean["السعر"] > 0) & (df_clean["المساحة"] > 0)].copy()

print("Before:", len(df))
print("After:", len(df_clean))

## Transaction references

Repeated transaction reference numbers are checked as part of the data quality review.

In [ ]:
print("Repeated transaction references:", df_clean["الرقم المرجعي للصفقة"].duplicated().sum())

## Quarterly comparison

Transaction counts, total recorded values, means, and medians are compared between Q1 and Q2.

In [ ]:
quarter_summary = df_clean.groupby("quarter")["السعر"].agg(["count", "sum", "mean", "median"])

quarter_summary

## Regional activity

The number of recorded sales is compared across regions and quarters.

In [ ]:
region_counts = pd.crosstab(df_clean["المنطقة"],df_clean["quarter"])

region_counts.sort_values("Q1", ascending=False)

## Value by region

Total transaction values are grouped by region and quarter to compare the distribution of recorded sales value.

In [ ]:
region_values = df_clean.groupby(["المنطقة", "quarter"])["السعر"].sum()

region_values.unstack(fill_value=0)

## City-level activity

Records are grouped by region and city to avoid combining cities that share a name.

In [ ]:
city_counts = df_clean.groupby(["المنطقة", "المدينة", "quarter"]).size()

city_counts.unstack(fill_value=0).sort_values("Q1", ascending=False).head(20)

## Property classifications

Transaction count, mean, and median are compared for residential, commercial, and agricultural records.

In [ ]:
property_summary = df_clean.groupby(["تصنيف العقار", "quarter"])["السعر"].agg(["count", "mean", "median"])

property_summary

## High-value transactions

The five largest recorded transactions are reviewed because unusually large values can affect the mean.

In [ ]:
df_clean.nlargest(5, "السعر")[["quarter", "المنطقة", "المدينة", "تصنيف العقار", "السعر", "المساحة"]]

## Transactions by region: Q1 vs Q2

The chart provides a quick comparison of recorded transaction activity across regions.

In [ ]:
region_counts.plot(kind="bar", figsize=(12, 5), title="Real Estate Records: Q1 vs Q2")

## Power BI dataset

Region labels and Gregorian date formatting are standardized before exporting `real_estate_clean.csv` for the dashboard and prediction notebook.

In [ ]:
df_clean["المنطقة"] = df_clean["المنطقة"].str.replace("منطقة ", "", regex=False)

dates = pd.to_datetime(df_clean["تاريخ الصفقة ميلادي"], format="%Y/%m/%d")
df_clean["تاريخ الصفقة ميلادي"] = (
    dates.dt.month.astype(str) + "/" + dates.dt.day.astype(str) + "/" + dates.dt.year.astype(str)
)

df_clean.to_csv("../data/real_estate_clean.csv", index=False, encoding="utf-8-sig", float_format="%.15g")
print("Saved data/real_estate_clean.csv")

## Key findings

- **Q1:** 59,099 records and approximately **SAR 70.63B** in total transaction value.
- **Q2:** 48,271 records and approximately **SAR 57.49B** in total transaction value.
- The number of records declined by **18.3%**, while total transaction value declined by **18.6%**.
- Median transaction value increased from **SAR 453,750** to **SAR 500,000**. Mean value was around **SAR 1.19M** in both quarters.

These are comparisons of recorded sales. They do not measure changes in prices for comparable properties.